# Parte 1: DCGAN de dragones (v3, Colab con GPU)

**Universo:** dragones de fantasía medieval con escamas, cuernos y alas.

Este notebook ejecuta la parte 1 completa sobre una GPU de Colab:
datos (965 dragones), las tres variantes controladas (base, pérdida y estabilización),
el informe comparativo, la elección del modelo, 200 candidatos y la galería regenerable.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)*.
Los resultados se guardan en Google Drive (`MyDrive/proyecto2_gan`). Si la sesión se corta,
volver a ejecutar desde arriba: el entrenamiento se reanuda desde el último checkpoint.

Antecedente (v2): con 22 imágenes y 900 actualizaciones las tres variantes colapsaron
(las 16 muestras de ruido fijo eran casi idénticas y D(fake)≈0.01). Ver `docs/resultados_fullbody_v2.md`.
v3 cambia los datos (965 imágenes + volteo horizontal) y el presupuesto (≈9 600 actualizaciones);
la estabilización pasa a ser normalización espectral en D.

## 1. Entorno, repositorio y Drive

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/DannyEst6109/Proyecto-2---GAN---Deep-Learning.git'
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/proyecto2_gan')
    ROOT = Path('/content/Proyecto-2---GAN---Deep-Learning')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    cwd = Path.cwd().resolve()
    ROOT = next(p for p in (cwd, *cwd.parents) if (p / 'dragon_gan').is_dir())
    WORK = ROOT
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
WORK.mkdir(parents=True, exist_ok=True)

import torch, torchvision
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Repositorio:', ROOT, '| commit:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
      capture_output=True, text=True).stdout.strip())
print('Resultados en:', WORK)
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__, '| dispositivo:', DEVICE,
      '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'sin GPU')
if DEVICE == 'cpu':
    print('AVISO: sin GPU cada época tarda ~30 s (≈5 h por variante). Cambiar el entorno a GPU.')


def run(*args):
    """Ejecuta un comando mostrando su salida en vivo y falla si el comando falla."""
    process = subprocess.Popen([sys.executable, '-X', 'utf8', *map(str, args)], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8')
    for line in process.stdout:
        print(line, end='')
    if process.wait():
        raise RuntimeError(f'Falló: {args}')

## 2. Datos

Fuente: *Dino or Dragon?* (agrigorev, Kaggle, versión 1), licencia CC0. Son imágenes sintéticas
de Stable Diffusion v1.4 (no son fotogramas de ninguna serie). Se usan las 965 imágenes de dragón
(train 767 + test 198), sin dinosaurios. El script verifica el hash del ZIP y de cada imagen.
Preparación: RGB, relleno proporcional a cuadrado, 64×64 con LANCZOS, normalización a [-1, 1];
aumento: volteo horizontal aleatorio durante el entrenamiento.

In [ ]:
import urllib.request
ZIP = WORK / 'raw' / 'dataset-v1.zip'
if not ZIP.exists():
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    url = 'https://www.kaggle.com/api/v1/datasets/download/agrigorev/dino-or-dragon?datasetVersionNumber=1'
    print('Descargando ~109 MB de Kaggle...')
    urllib.request.urlretrieve(url, ZIP)
DATA = ROOT / 'data' / 'dragons_all_v3'
run('scripts/prepare_dataset.py', '--archive', ZIP, '--selection', 'docs/data/curation_all_v3.csv', '--output', DATA)

In [ ]:
import json
import matplotlib.pyplot as plt
from torchvision.utils import make_grid
from dragon_gan.data import DragonDataset

CONFIG_PATH = ROOT / 'configs' / 'final_v3.json'
CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
dataset = DragonDataset(DATA, CONFIG['image_size'])
assert dataset.manifest()[1] == CONFIG['dataset_manifest_sha256'], 'Dataset distinto del declarado.'
generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(dataset), generator=generator)[:32].tolist()
grid = make_grid(torch.stack([dataset[i] for i in indices]), nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(12, 6)); plt.imshow(grid.permute(1, 2, 0)); plt.axis('off')
plt.title(f'Muestra del dataset a 64×64 ({len(dataset)} imágenes)')
plt.savefig(WORK / 'dataset_sample.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Configuración e hipótesis (escritas antes de entrenar)

Cada variante cambia **un solo factor** respecto de la base; datos, semilla, épocas,
inicialización de G, secuencia de z, orden de lotes, volteos y ruido fijo son idénticos.

In [ ]:
print(json.dumps(CONFIG, indent=2))
HYPOTHESES = ROOT / 'docs' / 'hypotheses' / 'v3'
for name in ('base', 'loss', 'stabilization'):
    print(f'\n[{name}]', (HYPOTHESES / f'{name}.txt').read_text(encoding='utf-8'))

## 4. Entrenamiento de las tres variantes

En una T4 cada variante debería tardar unos 10–20 min. Si Colab se desconecta, volver a ejecutar
las celdas 1–4: se detecta `plan.json` y se reanuda. `ONLY` permite ejecutar variantes por separado.

In [ ]:
RUNS = WORK / 'runs' / 'final_v3'
ONLY = ['base', 'loss', 'stabilization']
resume = ['--resume'] if (RUNS / 'plan.json').exists() else []
run('-m', 'scripts.run_experiments', '--config', CONFIG_PATH, '--data', DATA, '--output', RUNS,
    '--device', DEVICE, '--purpose', 'project', '--hypotheses', 'docs/hypotheses/v3',
    '--only', *ONLY, *resume)

## 5. Informe comparativo: curvas y rejilla de ruido fijo

In [ ]:
from IPython.display import Image as Show, display
existing = sorted(RUNS.glob('report*'))
REPORT = RUNS / f'report_{len(existing) + 1}'
run('-m', 'scripts.compare_experiments', '--runs', RUNS, '--output', REPORT)
display(Show(filename=str(REPORT / 'comparison.png')))
display(Show(filename=str(REPORT / 'fixed_noise_comparison.png')))

### Evolución detallada del ruido fijo por variante

Mismos 16 vectores z en varias épocas: permite ver cómo aparece la forma y si varias
posiciones empiezan a parecerse (colapso de modos).

In [ ]:
from PIL import Image as PILImage

def evolution(variant, epochs):
    rows = [PILImage.open(RUNS / variant / 'samples' / f'epoch_{e:04d}.png') for e in epochs]
    figure, axes = plt.subplots(1, len(rows), figsize=(3.2 * len(rows), 3.6))
    for axis, picture, epoch in zip(axes, rows, epochs):
        axis.imshow(picture); axis.set_title(f'{variant} · época {epoch}'); axis.axis('off')
    figure.tight_layout(); figure.savefig(WORK / f'evolution_{variant}.png', dpi=150); plt.show()

E = CONFIG['epochs']
EPOCHS = sorted({e for e in (0, 10, 50, E // 4, E // 2, 3 * E // 4, E) if e <= E})
for variant in ('base', 'loss', 'stabilization'):
    evolution(variant, EPOCHS)

In [ ]:
import csv, statistics
print(f"{'variante':14}{'D(real)':>10}{'D(fake)':>10}{'pérd. D':>10}{'desv. G':>10}{'ép. D<0.05':>12}   (últimas 50 épocas)")
for variant in ('base', 'loss', 'stabilization'):
    with (RUNS / variant / 'losses.csv').open(encoding='utf-8') as source:
        frame = [{k: float(v) for k, v in row.items()} for row in csv.DictReader(source)]
    last = frame[-50:]
    mean = lambda key: statistics.fmean(row[key] for row in last)
    spread = statistics.stdev(row['loss_g'] for row in last) if len(last) > 1 else 0.0
    print(f"{variant:14}{mean('d_real'):10.4f}{mean('d_fake'):10.4f}{mean('loss_d'):10.4f}{spread:10.4f}"
          f"{sum(row['loss_d'] < 0.05 for row in frame):12d}")

In [ ]:
# Diversidad dentro de cada snapshot: distancia media por píxel entre las 16 muestras de ruido fijo.
# Cerca de 0 = colapso (todas iguales). Es una señal auxiliar, no una métrica de calidad.
from dragon_gan.models import Generator
fixed_noise = torch.load(RUNS / 'base' / 'fixed_noise.pt', weights_only=True)

def load_generator(path):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    c = saved['config']
    g = Generator(c['latent_dim'], c['channels'], c['features'])
    g.load_state_dict(saved['generator']); g.eval()
    return g

def diversity(g):
    with torch.no_grad():
        images = g(fixed_noise).flatten(1)
    return torch.cdist(images, images).sum().item() / (len(images) * (len(images) - 1)) / images.shape[1] ** 0.5

plt.figure(figsize=(8, 4))
for variant in ('base', 'loss', 'stabilization'):
    paths = sorted((RUNS / variant / 'snapshots').glob('generator_epoch_*.pt'))
    values = [diversity(load_generator(path)) for path in paths]
    epochs = [int(path.stem[-4:]) for path in paths]
    plt.plot(epochs, values, marker='o', label=variant)
    print(variant, [f'{e}:{v:.3f}' for e, v in zip(epochs, values)])
plt.title('Diversidad del ruido fijo por snapshot'); plt.xlabel('Época'); plt.ylabel('distancia media')
plt.legend(); plt.savefig(WORK / 'diversity_snapshots.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Interpretación (completar con lo observado)

Escribir aquí, antes de elegir el modelo:

- **Base:** ¿aparecen dragones? ¿en qué época? ¿hay diversidad entre las 16 posiciones?
- **Pérdida (minimax vs. no saturante):** ¿se cumplió la hipótesis? Evidencia en curvas e imágenes.
- **Estabilización (normalización espectral):** ¿D(real)/D(fake) se alejan de 1/0? ¿más o menos diversidad?
- **Fallos observados:** colapso, artefactos de tablero, épocas donde empeora.
- **Modelo elegido y por qué** (por imágenes y diversidad, no por la pérdida mínima).

## 7. Elegir modelo y época

Comparar snapshots de una variante con el mismo ruido fijo. Elegir la época con dragones más
reconocibles **y** diversos (las GAN a veces empeoran al final).

In [ ]:
CHOSEN_VARIANT = 'stabilization'   # base, loss o stabilization
snapshots = sorted((RUNS / CHOSEN_VARIANT / 'snapshots').glob('generator_epoch_*.pt'))
figure, axes = plt.subplots(len(snapshots), 1, figsize=(16, 2.3 * len(snapshots)))
for axis, path in zip(axes, snapshots):
    with torch.no_grad():
        images = load_generator(path)(fixed_noise)
    axis.imshow(make_grid(images, nrow=16, normalize=True, value_range=(-1, 1)).permute(1, 2, 0))
    axis.set_ylabel(path.stem[-4:], rotation=0, labelpad=25); axis.set_xticks([]); axis.set_yticks([])
figure.tight_layout(); plt.show()

In [ ]:
CHOSEN_EPOCH = None   # None = último checkpoint (last.pt); o un número de snapshot, p. ej. 450
CHECKPOINT = (RUNS / CHOSEN_VARIANT / 'last.pt' if CHOSEN_EPOCH is None
              else RUNS / CHOSEN_VARIANT / 'snapshots' / f'generator_epoch_{CHOSEN_EPOCH:04d}.pt')
print('Modelo elegido:', CHECKPOINT)

## 8. Generar 200 candidatos

Semillas 1000–1199; el manifiesto guarda semilla, vector z y hash del checkpoint de cada PNG.
Los 200 son el denominador de la tasa de selección.

In [ ]:
from dragon_gan.generate import generate
tag = f'{CHOSEN_VARIANT}_{"last" if CHOSEN_EPOCH is None else CHOSEN_EPOCH}'
CANDIDATES = WORK / f'candidates_{tag}'
if not (CANDIDATES / 'manifest.json').exists():
    generate(CHECKPOINT, CANDIDATES, count=200, seed=1000)
SHEETS = WORK / f'contact_sheets_{tag}'
if not SHEETS.exists():
    run('scripts/candidate_contact_sheet.py', '--candidates', CANDIDATES, '--output', SHEETS)
for page in sorted(SHEETS.glob('page_*.png')):
    display(Show(filename=str(page)))

## 9. Galería final regenerable (con persona 2)

Elegir 10 personajes **distintos** después de la prueba de vecinos más cercanos.
Escribir los números de archivo y el criterio; la celda regenera los PNG desde los vectores z
y comprueba que son idénticos a los candidatos.

In [ ]:
SELECTED = []   # p. ej. [3, 17, 42, ...]: números de dragon_XXXX.png
CRITERIA = 'Escribir el criterio de selección (reconocible, distinto de los demás, no copia del vecino más cercano).'
if len(SELECTED) >= 10:
    import shutil
    source = json.loads((CANDIDATES / 'manifest.json').read_text(encoding='utf-8'))
    chosen = [row for row in source['characters'] if int(row['file'][7:11]) in SELECTED]
    assert len(chosen) == len(SELECTED)
    selection = dict(source, characters=chosen, selection_criteria=CRITERIA, status='final_gallery',
                     source_generated_count=len(source['characters']))
    selection_path = WORK / 'galeria_selection.json'
    selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')
    GALLERY = WORK / 'galeria'
    shutil.rmtree(GALLERY, ignore_errors=True)  # Regenerable desde los vectores z.
    generate(CHECKPOINT, GALLERY, manifest=selection_path)
    for row in chosen:
        assert (GALLERY / row['file']).read_bytes() == (CANDIDATES / row['file']).read_bytes()
    print(f'Galería regenerada e idéntica. Tasa de selección: {len(chosen)} de {len(source["characters"])}.')
else:
    print('Pendiente: elegir al menos 10 candidatos con persona 2.')

## 10. Paquete para persona 2

ZIP con informe, curvas, rejillas, candidatos, hojas de contacto, figuras y el generador elegido.
Los checkpoints completos (`last.pt`) quedan en Drive.

In [ ]:
import shutil, zipfile
PACKAGE = WORK / f'parte1_v3_{tag}.zip'
with zipfile.ZipFile(PACKAGE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUNS.rglob('*'):
        if path.is_file() and path.suffix not in ('.pt', '.tmp'):
            archive.write(path, 'runs/' + path.relative_to(RUNS).as_posix())
    for folder in (CANDIDATES, SHEETS):
        for path in folder.rglob('*'):
            archive.write(path, folder.name + '/' + path.relative_to(folder).as_posix())
    for path in WORK.glob('*.png'):
        archive.write(path, 'figuras/' + path.name)
    archive.write(CHECKPOINT, 'modelo_elegido/' + CHECKPOINT.name)
    archive.write(RUNS / 'base' / 'fixed_noise.pt', 'modelo_elegido/fixed_noise.pt')
print(PACKAGE, round(PACKAGE.stat().st_size / 1e6, 1), 'MB')
if IN_COLAB:
    from google.colab import files
    files.download(str(PACKAGE))